# Fase 2 — Cálculo de métricas

Lee los archivos `.pkl` de cada corrida y calcula **HV**, **GD** e **IGD**.

- **Frente de referencia:** unión global de todos los `fitnessF`, no-dominados.
- **Punto de referencia HV:** `[0, 0]` (consistente con el TFG).
- **Salida:** `results/metrics.csv`

In [1]:
import os
import glob
import pickle
import re
import numpy as np
import pandas as pd

from pymoo.indicators.hv import Hypervolume
from pymoo.indicators.gd import GD
from pymoo.indicators.igd import IGD
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

RESULTS_BASE = './results'
N_VALUES     = [50, 100, 150, 200]
VARIANTES    = ['original', 'modified']
SEED_MODES   = ['same_seed', 'diff_seed']

## 1. Cargar todos los archivos `.pkl`

In [2]:
def load_all_runs():
    runs = []
    for variante in VARIANTES:
        for seed_mode in SEED_MODES:
            for n in N_VALUES:
                folder = os.path.join(RESULTS_BASE, variante, seed_mode, f'N{n}')
                if not os.path.exists(folder):
                    continue
                pkls = sorted(glob.glob(os.path.join(folder, '*.pkl')))
                for i, pkl_path in enumerate(pkls):
                    with open(pkl_path, 'rb') as f:
                        data = pickle.load(f)
                    m = re.search(r'seed_(\d+)_', os.path.basename(pkl_path))
                    seed = int(m.group(1)) if m else None
                    runs.append({
                        'variante':  variante,
                        'seed_mode': seed_mode,
                        'N':         n,
                        'corrida':   i + 1,
                        'seed':      seed,
                        'fitnessF':  np.array(data['fitnessF']),
                    })
    return runs

runs = load_all_runs()
print(f'Total de corridas cargadas: {len(runs)}')

summary = pd.DataFrame([
    {'variante': r['variante'], 'seed_mode': r['seed_mode'], 'N': r['N']}
    for r in runs
]).groupby(['variante', 'seed_mode', 'N']).size().rename('corridas')
print(summary.to_string())

Total de corridas cargadas: 119
variante  seed_mode  N  
original  diff_seed  50     30
                     100    30
                     150    30
                     200    29


## 2. Construir frente de referencia global

In [3]:
all_F = np.vstack([r['fitnessF'] for r in runs])
nds   = NonDominatedSorting()
front_idx = nds.do(all_F, only_non_dominated_front=True)
ref_front = all_F[front_idx]

print(f'Puntos totales (unión): {len(all_F)}')
print(f'Tamaño frente de referencia (no-dominados): {len(ref_front)}')

Puntos totales (unión): 14800
Tamaño frente de referencia (no-dominados): 2211


## 3. Calcular HV, GD e IGD por corrida

In [4]:
hv_indicator  = Hypervolume(ref_point=np.zeros(2))
gd_indicator  = GD(ref_front)
igd_indicator = IGD(ref_front)

records = []
for r in runs:
    F = r['fitnessF']
    records.append({
        'variante':  r['variante'],
        'seed_mode': r['seed_mode'],
        'N':         r['N'],
        'corrida':   r['corrida'],
        'seed':      r['seed'],
        'HV':        hv_indicator.do(F),
        'GD':        gd_indicator.do(F),
        'IGD':       igd_indicator.do(F),
    })

metrics_df = pd.DataFrame(records)
out_path = os.path.join(RESULTS_BASE, 'metrics.csv')
metrics_df.to_csv(out_path, index=False)
print(f'Guardado: {os.path.abspath(out_path)}')
metrics_df.head()

Guardado: c:\Projects\tfg\src\results\metrics.csv


,variante,seed_mode,N,corrida,seed,HV,GD,IGD
0,original,diff_seed,50,1,10263,0.563527,0.003746,0.010321
1,original,diff_seed,50,2,18588,0.565530,0.001934,0.009645
2,original,diff_seed,50,3,18667,0.397160,0.081476,0.109410
3,original,diff_seed,50,4,1894,0.566633,0.002374,0.008473
4,original,diff_seed,50,5,22806,0.565236,0.002239,0.009806


## 4. Resumen estadístico por grupo

In [5]:
metrics_df.groupby(['variante', 'seed_mode', 'N'])[['HV', 'GD', 'IGD']].agg(['mean', 'std', 'median'])

HV                            GD            \
                            mean       std    median      mean       std   
variante seed_mode N                                                       
original diff_seed 50   0.542304  0.057972  0.563718  0.013591  0.027930   
                   100  0.573364  0.000606  0.573466  0.001547  0.000387   
                   150  0.576277  0.000503  0.576263  0.001043  0.000327   
                   200  0.577545  0.001399  0.577745  0.000897  0.000737   

                                       IGD                      
                          median      mean       std    median  
variante seed_mode N                                            
original diff_seed 50   0.002907  0.023192  0.034518  0.010144  
                   100  0.001511  0.004627  0.000253  0.004530  
                   150  0.000989  0.003050  0.000210  0.003028  
                   200  0.000747  0.002378  0.000928  0.002212